In [ ]:
%load_ext autoreload
%autoreload 2

from src.utils.viz import display_grayscale

In [ ]:
# Layer 1 : independent small tracings from a segmentation model
# Already done

In [ ]:
# Layer 2 : combine individual small tracings into a large tracing for the original image
# Functionnal, could be improved
# Done : caching / storing tracings in .tif file at some specific location

In [ ]:
from src.tracers.DLTracer import DLTracer
from src.NNs.Unet import UNetModel
model_path = r"./data/trained_models/topology_aware_models/2025-07-21_AugDataset_thick_axons_connected_loss_0.4_0.0.pth"
tracer = DLTracer(model_path, UNetModel, 128, tracer_name="test_tracer")

In [ ]:
from src.utils.imageio import tif_to_numpy
import numpy as np
# path = r".\data\project_scans\rat301\b516\ipsi_inner\Ipsilesional_ROI.tif"
path = r".\data\project_scans\rat301\b468\contra_inner\Contralesional_ROI.tif"
test_image = tif_to_numpy(path)
test_image = test_image[:, :, :1]

In [ ]:
test_image.shape

In [ ]:
from src.utils.imageio import generate_image_outer_mask
mask = generate_image_outer_mask(test_image)
display_grayscale(mask, title="ROI mask")

In [ ]:
tracing_cache_file = 'test_me/'
trace = tracer.trace(test_image, mask, tracing_cache_folder=tracing_cache_file).squeeze()

In [ ]:
trace.shape

In [ ]:
from src.utils.trace_manips import thicken_trace
from src.imgproc.utils import contrast_img
display_test_image = test_image.copy()
display_test_image = contrast_img(display_test_image, contrast_factor=6)
display_test_image[~mask] = np.nan
display_grayscale(display_test_image, title="Original ROI Example")

display_trace = thicken_trace(trace, 7).astype(np.float32)
display_trace[~mask] = np.nan
display_grayscale(display_trace, title="Automatic Image Segmentation")

In [ ]:
# Layer 3 : Get the large tracing and infer features 
# Done. Currently, features infered from small images. (First tile, then maybe something else)

In [ ]:
from src.image_extractors.BaselineMeanExtractors import PopulationMeanExtractor, ImageMeanExtractor
from src.image_extractors.ThresholdDensityExtractor import ThresholdDensityExtractor
from src.image_extractors.TracerExtractor import TraceExtractor

import numpy as np

In [ ]:
from src.utils.traceProps import get_axon_count, get_mean_axon_length, get_trace_density
ground_truth_functions = [get_axon_count, get_mean_axon_length, get_trace_density]
feature_names = ["Fibre Count", "Mean Fibre Length", "Foreground Density"]

extractors = [
    PopulationMeanExtractor(),
    ImageMeanExtractor(feature_extraction_tile_size=64),
    ThresholdDensityExtractor(feature_extraction_tile_size=64, local=True),
    TraceExtractor(tracer, ground_truth_functions=ground_truth_functions, 
                   feature_names=feature_names, feature_extraction_tile_size=64),
    # Add others if needed
]



In [ ]:
[ex.is_fittable for ex in extractors]

In [ ]:

# reduce to non-fit ones for testing
extractors = [ex for ex in extractors if not ex.is_fittable]


In [ ]:
extractors

In [ ]:
for extractor in extractors:
    features = extractor.extract(test_image, mask, cache_folder="test_me")
        
    print(f"Extractor {extractor.extractor_name} created features of shape : {features.shape}")
    
    for feature_image, feature_name in zip(np.moveaxis(features, 2, 0), extractor.feature_names):
        print(f"Showing feature image for feature {feature_name}")
        display_grayscale(feature_image, title=f"Feature : {feature_name}")

In [ ]:
# Layer 4 : Make a features -> properties model that can fit data with input (image, [properties]) pairs.
# Done

In [ ]:
from src.image_extractors.PropertyModel import PropertyModel
from sklearn.linear_model import LinearRegression

cache_path = r'.\data\project_tracing_and_features\test_model'
prop_model = PropertyModel(extractors=extractors, model=LinearRegression(), cache_folder=cache_path)

In [ ]:
from src.evaluation.Trainer import Trainer
dataset_path = r'.\data\manual_tracings\2025-05-31_rat301_train'
trainer = Trainer(dataset_path, ground_truth_functions=ground_truth_functions)
trainer.fit_model(prop_model)

In [ ]:
image_path = r".\data\project_scans\rat301\b468\contra_inner\\doesntmatter.tif"
properties = prop_model.predict(test_image, mask=mask, tile_size=64, image_path=image_path)

In [ ]:
propery_names = ["Axon Count", "Mean Axon Length", "Axon innervation density"]
prop_model.show_feature_weights(propery_names)

In [ ]:
print(f"Showing feature image for feature property_map")
display_grayscale(properties[:,:, 0], title="Predicted Axon Count")
display_grayscale(properties[:,:, 1], title="Predicted Mean Axon Length")
display_grayscale(properties[:,:, 2], title="Predicted Axon Density")

In [ ]:
# Layer 5: Evaluation by first inferring on whole image, then obtaining predictions by cropping features to specific starting point
# Done, not tested

from src.evaluation.RegressionEvaluator import RegressionEvaluator
ground_truth_functions = [get_axon_count, get_mean_axon_length, get_trace_density]
propery_names = ["Axon Count", "Mean Axon Length", "Axon innervation density"]
dataset_path = r'.\data\manual_tracings\2025-05-31_rat301_test'
tracings_cache_folder = r'.\data\tracings\project_tracing_and_features'
evaluator = RegressionEvaluator(image_paths=dataset_path, tracings_cache_folder=tracings_cache_folder, 
                                estimated_names=propery_names, ground_truth_functions=ground_truth_functions)


In [ ]:
evaluator.display_random_ROIS(n_rois=1)

In [ ]:
evaluator.display_random_images_and_predictions(prop_model, n_images=10)

In [ ]:
evaluator.evaluate_ROIs(prop_model, display_fitness=True)